# Block 5 — Review queue and LIS order

HiTL items from Block 4. Human/nurse patches edit **drafts**, then Block 4 constraints run again. LLM suggestions (optional) never auto-apply.

Do **not** upload clinic PHI.


## 0. Download src/med_doc (zipball)


In [ ]:
# BOOTSTRAP_V6 — always refresh zipball (stale /content/epq3 lacks new kwargs like output_mode)
import importlib
import inspect
import os
import shutil
import sys
import urllib.request
import zipfile
from pathlib import Path

CONTENT = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO = CONTENT / "epq3"
SRC = REPO / "src"
URL = "https://codeload.github.com/RwaRwa599/epq3/zip/refs/heads/block1"

zpath = CONTENT / "epq3-block1.zip"
print("Downloading", URL)
urllib.request.urlretrieve(URL, zpath)
extract = CONTENT / "_epq3_extract"
if extract.exists():
    shutil.rmtree(extract)
extract.mkdir()
with zipfile.ZipFile(zpath) as zf:
    zf.extractall(extract)
found = list(extract.glob("*/src/med_doc/__init__.py"))
if not found:
    raise RuntimeError(f"zip missing src/med_doc: {list(extract.iterdir())}")
unpacked = found[0].parents[2]
if REPO.exists():
    shutil.rmtree(REPO)
shutil.move(str(unpacked), str(REPO))
shutil.rmtree(extract, ignore_errors=True)
zpath.unlink(missing_ok=True)

src = str(SRC.resolve())
while src in sys.path:
    sys.path.remove(src)
sys.path.insert(0, src)
os.chdir(REPO)
for name in list(sys.modules):
    if name == "med_doc" or name.startswith("med_doc."):
        del sys.modules[name]
importlib.invalidate_caches()
import med_doc
from med_doc.pipeline import run_blocks_1_to_5

print("BOOTSTRAP_V6")
print("cwd:", os.getcwd())
print("med_doc:", med_doc.__file__)
print("params:", list(inspect.signature(run_blocks_1_to_5).parameters))
from med_doc.htr.marks import TICK_POLICY
print("tick_policy:", TICK_POLICY)
if "output_mode" not in inspect.signature(run_blocks_1_to_5).parameters:
    raise RuntimeError(
        "stale med_doc (no output_mode). Runtime → Disconnect and delete runtime, "
        "re-open Run_in_Colab.ipynb from GitHub branch block1, then Run all."
    )
if TICK_POLICY != "slash-v2":
    raise RuntimeError(
        f"stale med_doc tick_policy={TICK_POLICY!r}. Disconnect and delete runtime, "
        "re-open Run_in_Colab.ipynb from GitHub branch block1."
    )


In [ ]:
# Runtime deps via pip CLI (not %pip / not pip -e — those restart Colab mid-run).
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless", "pydantic", "matplotlib", "Pillow", "numpy"]
)


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V6 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

from pathlib import Path
import json
import cv2
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

OUT = Path("/content/pipeline") if Path("/content").is_dir() else Path("outputs/colab_pipeline")
OUT.mkdir(parents=True, exist_ok=True)
OUTPUT_MODE = "dev"  # overlays + block5.zip; "user" = order.json only

def show_rgb(path, title="", figsize=(10, 8)):
    path = Path(path)
    if not path.exists():
        print("missing", path)
        return
    bgr = cv2.imread(str(path))
    if bgr is None:
        print("unreadable", path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=figsize)
    plt.imshow(rgb)
    plt.title(title or path.name)
    plt.axis("off")
    plt.show()

def download(path):
    path = Path(path)
    print(path, f"({path.stat().st_size / 1024:.1f} KB)" if path.exists() else "missing")
    if colab_files and path.exists():
        colab_files.download(str(path))

def demo_sheet() -> Path:
    from med_doc.paths import SYNTHETIC_DIR
    sheet = SYNTHETIC_DIR / "lab_request_v0_blank.png"
    assert sheet.exists(), sheet
    return sheet


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V6 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

from med_doc.htr.batch import process_from_block1
from med_doc.kg import KnowledgeGraph
from med_doc.normalization.batch import normalize_batch
from med_doc.rescoring import process_from_block3
from med_doc.review import ReviewPatch, process_from_block4

kg = KnowledgeGraph.load()

def ensure_block1():
    z = OUT / "block1.zip"
    if z.exists():
        return z
    return Path(normalize_batch([demo_sheet()], output_dir=OUT / "b1", output_zip=z)["output_zip"])

def ensure_block3():
    z = OUT / "block3.zip"
    if z.exists():
        return z
    b1 = ensure_block1()
    return Path(process_from_block1(b1, output_dir=OUT / "b3", output_zip=z, kg=kg, backend="lexicon", mode="both")["output_zip"])

def ensure_block4():
    z = OUT / "block4.zip"
    if z.exists():
        return z
    b3 = ensure_block3()
    return Path(process_from_block3(b3, output_dir=OUT / "b4", output_zip=z, kg=kg)["output_zip"])


## 1. Queue without patches


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V6 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

b5_open = process_from_block4(ensure_block4(), output_dir=OUT / "b5_open", kg=kg)
doc_id = b5_open["manifest"]["documents"][0]["doc_id"]
review = json.loads((OUT / "b5_open" / "docs" / doc_id / "review.json").read_text())
order = json.loads((OUT / "b5_open" / "docs" / doc_id / "order.json").read_text())
print("auto_passed", review["auto_passed"], "queue", [i["field_id"] for i in review["queue"]])
print("needs_review", order["needs_review"], "observed", order["observed_tubes"])


## 2. Nurse override (example: EDTA count = 1) then commit


In [ ]:
import os
import sys
from pathlib import Path

def _guard():
    content = Path("/content") if Path("/content").is_dir() else Path.cwd()
    hits = list(content.glob("epq3/src/med_doc/__init__.py"))
    hits += list(content.glob("*/src/med_doc/__init__.py"))
    if not hits:
        raise ModuleNotFoundError(
            "med_doc missing. Run the FIRST code cell until it prints BOOTSTRAP_V6 "
            "and med_doc: .../src/med_doc/__init__.py. Open Run_in_Colab.ipynb from "
            "GitHub branch block1. Runtime → Disconnect and delete runtime, then Run all."
        )
    src = hits[0].parents[1]
    root = hits[0].parents[2]
    os.chdir(root)
    sp = str(src.resolve())
    if sp not in sys.path:
        sys.path.insert(0, sp)
    return sp
_guard()

reviews = {doc_id: [ReviewPatch(field_id="tube_edta", action="set_tube", value="1")]}
b5 = process_from_block4(
    ensure_block4(),
    output_dir=OUT / "b5",
    output_zip=OUT / "block5.zip",
    kg=kg,
    reviews=reviews,
    output_mode=OUTPUT_MODE,
)
print("mode", b5["output_mode"], "json", b5["output_json"])
bundle = json.loads(Path(b5["output_json"]).read_text())
order = bundle["orders"][0]
print("committed observed", order["observed_tubes"])
print("needs_review", order["needs_review"], "valid", order["is_valid"])
if b5["output_mode"] == "dev":
    hyp = json.loads((OUT / "b5" / "docs" / order["doc_id"] / "hypotheses.json").read_text())
    print("OCR tube still", hyp["verbal"].get("tube_edta", {}).get("raw_text"), hyp["verbal"].get("tube_edta", {}).get("source"))
    download(OUT / "block5.zip")
else:
    download(Path(b5["output_json"]))
